# Multi-Agent SAC (MARL) — Independent Learners cho Smart Grid

**Giai đoạn 1** trong lộ trình phát triển: `Centralized SAC → MARL (IL) → Federated RL`

## Kiến trúc tổng quan

```
Centralized (baseline):              MARL — Independent Learners:
        Agent                    Agent₁   Agent₂  ...  Agent₆
          ↓                        ↓        ↓             ↓
  obs ∈ ℝ²⁴⁶                  obs₁∈ℝ⁴¹  obs₂∈ℝ⁴¹   obs₆∈ℝ⁴¹
  act ∈ ℝ¹⁸                   act₁∈ℝ³   act₂∈ℝ³    act₆∈ℝ³
          ↓                        ↓        ↓             ↓
  [B1,B2,B3,B4,B5,B6]            [B1]     [B2]         [B6]
```

## Lý thuyết toán học

### 1. Bài toán Dec-POMDP
MARL được mô hình hóa như **Decentralized Partially Observable MDP (Dec-POMDP)**:
$$\\langle \\mathcal{I}, \\mathcal{S}, \\{\\mathcal{A}^i\\}, \\mathcal{P}, \\{\\mathcal{R}^i\\}, \\{\\Omega^i\\}, \\mathcal{O}, \\gamma \\rangle$$

| Ký hiệu | Ý nghĩa |
|---|---|
| $\\mathcal{I} = \\{1,...,N\\}$ | Tập N agent (N = 6 tòa nhà) |
| $s \\in \\mathcal{S}$ | Trạng thái toàn cục (ẩn) |
| $o^i \\in \\Omega^i$ | Quan sát cục bộ của agent $i$ (41 chiều) |
| $a^i \\in \\mathcal{A}^i$ | Hành động của agent $i$ (3 chiều) |
| $r^i$ | Phần thưởng agent $i$ nhận được |

### 2. Independent Learners (IL)
Mỗi agent $i$ tự học policy $\\pi^i$ **độc lập**, coi các agent khác là một phần của môi trường:

$$\\pi^i_* = \\arg\\max_{\\pi^i} \\mathbb{E}_{\\tau \\sim \\pi^i} \\left[ \\sum_{t=0}^T r^i_t + \\alpha \\mathcal{H}(\\pi^i(\\cdot|o^i_t)) \\right]$$

**Ưu điểm:**
- Observation space nhỏ hơn: $41$ vs $246$ chiều → hội tụ nhanh hơn
- Scalable: thêm nhà mới không cần train lại toàn bộ
- Nền tảng tự nhiên cho **Federated RL** (thêm aggregation là xong)

**Hạn chế (Non-stationarity problem):**
Khi agent $i$ đang học, môi trường **thay đổi** vì các agent $j \\neq i$ cũng cập nhật policy → vi phạm giả định Markov. Đây là vấn đề cốt lõi mà FedRL giải quyết qua periodic aggregation.

### 3. Quan hệ với Centralized SAC
```
Centralized SAC = MARL với shared policy và joint obs/action
IL-MARL        = N SAC độc lập, mỗi cái có policy riêng
FedRL          = IL-MARL + periodic weight aggregation
```

In [1]:
# Cài đặt trực tiếp vào hệ thống Colab, bỏ qua Drive
print("⏳ Đang cài đặt thư viện vào hệ thống Colab, vui lòng chờ...")

!pip install --no-cache-dir \
    "numpy<2.0.0" \
    "pandas>=2.1.0" \
    "torch>=2.2.0" \
    "citylearn==2.5.0" \
    "stable-baselines3>=2.2.1" \
    "gymnasium==0.28.1" \
    "tensorboard"

print("\n✅ Đã cài xong! Bạn có thể import và sử dụng ngay lập tức.")

⏳ Đang cài đặt thư viện vào hệ thống Colab, vui lòng chờ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 37.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 255.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of stable-baselines3 to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.5/398.5 kB 435.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 440.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 398.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 413.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.


✅ Đã cài xong! Bạn có thể import và sử dụng ngay lập tức.


In [2]:
!nproc

8


In [1]:
import os
import glob
import copy
from pathlib import Path
import numpy as np
import pandas as pd
import subprocess

# SB3 Imports
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback, BaseCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.monitor import Monitor

# CityLearn Imports
from citylearn.citylearn import CityLearnEnv
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper


# ================================================================
# Custom callback: rsync local cache → Drive định kỳ trong khi training
# ================================================================
class RsyncToDriveCallback(BaseCallback):
    """Sync local checkpoint dir lên Drive mỗi N steps (background)."""
    def __init__(self, local_dir, drive_dir, rsync_every_steps, verbose=0):
        super().__init__(verbose)
        self.local_dir = str(local_dir)
        self.drive_dir = str(drive_dir)
        self.rsync_every_steps = rsync_every_steps
        self.last_rsync_step = 0

    def _on_step(self):
        if self.num_timesteps - self.last_rsync_step >= self.rsync_every_steps:
            try:
                subprocess.run(
                    ["rsync", "-a", "--quiet",
                     f"{self.local_dir}/", f"{self.drive_dir}/"],
                    check=False, timeout=120
                )
                self.last_rsync_step = self.num_timesteps
                if self.verbose:
                    print(f"  💾 Synced to Drive @ step {self.num_timesteps:,}")
            except Exception as e:
                if self.verbose:
                    print(f"  ⚠ Rsync failed: {e}")
        return True

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# ================================================================
# MOUNT GOOGLE DRIVE & SETUP PATHS (chạy đầu tiên!)
# ================================================================
from google.colab import drive

# Mount Drive nếu chưa mount
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("✓ Drive đã mount sẵn")

# ⭐ Thư mục đích trên Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/BigData/xxxmarl_solar_comfort')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Local cache để training nhanh (Drive chậm cho I/O liên tục)
LOCAL_CACHE = Path('/content/_marl_solar_cache')
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

print(f"📁 Drive folder : {DRIVE_ROOT}")
print(f"📁 Local cache  : {LOCAL_CACHE}")

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ KHÔNG có GPU! Training sẽ rất chậm.")

Mounted at /content/drive
📁 Drive folder : /content/drive/MyDrive/BigData/xxxmarl_solar_comfort
📁 Local cache  : /content/_marl_solar_cache
✅ GPU: Tesla T4 (15.6 GB)


In [4]:
# ==============================================================
# GLOBAL PARAMETERS (TỐI ƯU cho Colab Pro T4)
# ==============================================================
NUM_HOUSES       = 6
NUM_CPU          = 4         # ⬆️ 2→4: tăng env parallelism
EPISODE_LENGTH   = 168       # 1 tuần (giờ)

# SAC Hyperparameters
NET_ARCH         = [256, 256]
BATCH_SIZE       = 512       # ⬆️ 256→512: tận dụng GPU VRAM dư
BUFFER_SIZE      = 300_000
LEARNING_STARTS  = 1_000

# ⚡ Train scheduling — ép GPU làm việc nhiều hơn
TRAIN_FREQ       = 8        # ⬆️ Tăng cùng NUM_CPU
GRADIENT_STEPS   = 16        # ⬆️ Bù sample efficiency

TOTAL_TIMESTEPS  = 600_000   # ⬇️ 600k→200k: SAC hội tụ tầm 100-200k là đủ
SAVE_EVERY_EP    = 100        # ⬇️ 200→50: lưu checkpoint thường xuyên hơn

# ⭐ Tần suất sync local → Drive (chạy song song training)
RSYNC_EVERY_EP   = 200

BASE_SCHEMA      = "citylearn_challenge_2023_phase_3_1"
SESSION_NAME     = "marl_sac_solar_comfort"   # ⭐ Reward mới: SolarPenaltyAndComfortReward

# ⭐ Paths: training trên local cache (nhanh), backup lên Drive (bền vững)
TENSORBOARD_DIR  = LOCAL_CACHE / "training_logs"
MODEL_DIR        = LOCAL_CACHE / "models" / SESSION_NAME
DRIVE_MODEL_DIR  = DRIVE_ROOT / "models" / SESSION_NAME
DRIVE_TB_DIR     = DRIVE_ROOT / "training_logs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
TENSORBOARD_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_TB_DIR.mkdir(parents=True, exist_ok=True)

# ⭐ Resume: nếu Drive đã có checkpoints → rsync về local
if any(DRIVE_MODEL_DIR.iterdir()):
    print("Đang đồng bộ checkpoints từ Drive về local cache (resume)...")
    subprocess.run(
        ["rsync", "-a", f"{DRIVE_MODEL_DIR}/", f"{MODEL_DIR}/"],
        check=False
    )
    print("✓ Sync xong, sẵn sàng resume")

print(f"{'='*62}")
print(f"  MARL — Independent Learners (TỐI ƯU)")
print(f"{'='*62}")
print(f"  Agents              : {NUM_HOUSES}")
print(f"  Workers/agent       : {NUM_CPU} (DummyVecEnv)")
print(f"  Steps/agent         : {TOTAL_TIMESTEPS:,}")
print(f"  Total steps         : {NUM_HOUSES * TOTAL_TIMESTEPS:,}")
print(f"  Batch / Train freq  : {BATCH_SIZE} / {TRAIN_FREQ}")
print(f"  Gradient steps      : {GRADIENT_STEPS}")
print(f"  Save checkpoint     : mỗi {SAVE_EVERY_EP} eps (~{(EPISODE_LENGTH * SAVE_EVERY_EP)//NUM_CPU:,} steps)")
print(f"  Rsync to Drive      : mỗi {RSYNC_EVERY_EP} eps")
print(f"  Drive folder        : {DRIVE_MODEL_DIR}")
print(f"{'='*62}")
print(f"  📊 ETA dự kiến: ~3-5 giờ (trước đó: ~12h)")
print(f"  💡 Theo dõi: !nvidia-smi để xem GPU usage")
print(f"{'='*62}")

Đang đồng bộ checkpoints từ Drive về local cache (resume)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✓ Sync xong, sẵn sàng resume
  MARL — Independent Learners (TỐI ƯU)
  Agents              : 6
  Workers/agent       : 4 (DummyVecEnv)
  Steps/agent         : 600,000
  Total steps         : 3,600,000
  Batch / Train freq  : 512 / 8
  Gradient steps      : 16
  Save checkpoint     : mỗi 100 eps (~4,200 steps)
  Rsync to Drive      : mỗi 200 eps
  Drive folder        : /content/drive/MyDrive/BigData/xxxmarl_solar_comfort/models/marl_sac_solar_comfort
  📊 ETA dự kiến: ~3-5 giờ (trước đó: ~12h)
  💡 Theo dõi: !nvidia-smi để xem GPU usage


## Xây dựng môi trường per-building

Mỗi agent cần một **môi trường riêng** chỉ chứa tòa nhà của mình:

```
full_schema (6 buildings)
    ↓ filter
schema_B1  schema_B2  ...  schema_B6
    ↓          ↓                ↓
 Env_B1     Env_B2          Env_B6
    ↓          ↓                ↓
 SAC_B1     SAC_B2          SAC_B6
```

**NormalizedObservationWrapper** áp dụng min-max per feature:
$$\\hat{o}^i_k = \\frac{o^i_k - o^{i,min}_k}{o^{i,max}_k - o^{i,min}_k} \\in [0,1]$$

**SubprocVecEnv** chạy `NUM_CPU` bản copy song song để thu thập dữ liệu nhanh hơn, giảm variance gradient.

> **Lưu ý**: Mỗi agent train **độc lập** — không biết gì về hành động/trạng thái của các agent khác trong quá trình training. Đây là "Independent" trong Independent Learners.

In [5]:
def make_env(schema_dict, episode_time_steps):
    """
    Factory function tạo môi trường CityLearn cho 1 tòa nhà.
    Dùng cho SubprocVecEnv — mỗi worker là 1 bản copy độc lập.
    """
    def _init():
        env = CityLearnEnv(
            schema=schema_dict,
            central_agent=True,
            episode_time_steps=episode_time_steps
        )
        env = NormalizedObservationWrapper(env)   # obs → [0,1]
        env = StableBaselines3Wrapper(env)         # gym-compatible API
        env = Monitor(env)                         # ghi lại ep_reward, ep_len
        return env
    return _init


def get_single_building_schema(full_schema, building_name):
    """Lọc schema chỉ giữ lại 1 tòa nhà duy nhất."""
    schema = copy.deepcopy(full_schema)
    schema["buildings"] = {building_name: schema["buildings"][building_name]}
    return schema


# ------------------------------------------------------------------
# Tải schema gốc và lấy danh sách tòa nhà
# ------------------------------------------------------------------
print("Đang tải schema...")
_temp_env = CityLearnEnv(schema=BASE_SCHEMA)
FULL_SCHEMA = copy.deepcopy(_temp_env.schema)
ALL_BUILDING_NAMES = list(FULL_SCHEMA["buildings"].keys())
BUILDING_NAMES = ALL_BUILDING_NAMES[:NUM_HOUSES]
del _temp_env

print(f"Tất cả các tòa nhà trong dataset : {ALL_BUILDING_NAMES}")
print(f"Tòa nhà được chọn ({NUM_HOUSES} nhà): {BUILDING_NAMES}")

# Tạo per-building schema dict
BUILDING_SCHEMAS = {
    name: get_single_building_schema(FULL_SCHEMA, name)
    for name in BUILDING_NAMES
}

# ------------------------------------------------------------------
# Kiểm tra kích thước obs/action của 1 tòa nhà
# ------------------------------------------------------------------
_probe_env = make_env(BUILDING_SCHEMAS[BUILDING_NAMES[0]], EPISODE_LENGTH)()
OBS_DIM_PER_AGENT = _probe_env.observation_space.shape[0]
ACT_DIM_PER_AGENT = _probe_env.action_space.shape[0]
_probe_env.close()

print(f"\nPer-agent observation dim : {OBS_DIM_PER_AGENT}")
print(f"Per-agent action dim      : {ACT_DIM_PER_AGENT}")
print(f"Joint obs dim (6 agents)  : {OBS_DIM_PER_AGENT * NUM_HOUSES}")
print(f"Joint act dim (6 agents)  : {ACT_DIM_PER_AGENT * NUM_HOUSES}")

# ⭐ Áp reward function mới (SolarPenaltyAndComfortReward) vào MỌI building schema
for _bn, _sch in BUILDING_SCHEMAS.items():
    _sch["reward_function"]["type"] = "citylearn.reward_function.SolarPenaltyAndComfortReward"
print(f"✓ Reward function: SolarPenaltyAndComfortReward (áp cho {len(BUILDING_SCHEMAS)} buildings)")


Đang tải schema...


INFO:root:The dataset names DNE in cache. Will download from intelligent-environments-lab/CityLearn/tree/v2.5.0 GitHub repository and write to /root/.cache/citylearn/v2.5.0/dataset_names.json. Next time DataSet.get_dataset_names is called, it will read from cache unless DataSet.clear_cache is run first.
INFO:root:Go here /root/.cache/citylearn/v2.5.0/datasets/citylearn_challenge_2023_phase_3_1/schema.json 
INFO:root:The citylearn_challenge_2023_phase_3_1 dataset DNE in cache. Will download from intelligent-environments-lab/CityLearn/tree/v2.5.0 GitHub repository and write to /root/.cache/citylearn/v2.5.0/datasets. Next time DataSet.get_dataset('citylearn_challenge_2023_phase_3_1') is called, it will read from cache unless DataSet.clear_cache is run first.
INFO:root:The PV sizing data DNE in cache. Will download from intelligent-environments-lab/CityLearn/tree/v2.5.0 GitHub repository and write to /root/.cache/citylearn/v2.5.0/misc. Next time DataSet.get_pv_sizing_data is called, it wil

Tất cả các tòa nhà trong dataset : ['Building_1', 'Building_2', 'Building_3', 'Building_4', 'Building_5', 'Building_6']
Tòa nhà được chọn (6 nhà): ['Building_1', 'Building_2', 'Building_3', 'Building_4', 'Building_5', 'Building_6']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Per-agent observation dim : 32
Per-agent action dim      : 3
Joint obs dim (6 agents)  : 192
Joint act dim (6 agents)  : 18
✓ Reward function: SolarPenaltyAndComfortReward (áp cho 6 buildings)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Khởi tạo các SAC Agent

Mỗi agent là một **SAC độc lập** với:
- Policy network: `[256, 256]` (giống `single_house_sac` baseline để so sánh công bằng)
- Replay buffer riêng: `100,000` transitions
- Entropy coefficient $\\alpha$ tự điều chỉnh

**Checkpoint resume**: Nếu đã có checkpoint từ lần train trước, tự động tiếp tục từ đó (kể cả replay buffer).

### Cấu trúc thư mục lưu model:
```
models/marl_sac_independent_learners/
├── Building_1/
│   ├── sac_Building_1_XXXXX_steps.zip
│   ├── sac_Building_1_XXXXX_steps_replay_buffer.pkl
│   └── sac_Building_1_final.zip
├── Building_2/
│   └── ...
└── Building_6/
    └── ...
```

In [6]:
agents    = {}   # {building_name: SAC model | None nếu skip}
vec_envs  = {}   # {building_name: VecEnv | None nếu skip}

for building_name in BUILDING_NAMES:
    print(f"\n[{building_name}] Đang khởi tạo...")

    agent_dir = MODEL_DIR / building_name
    agent_dir.mkdir(parents=True, exist_ok=True)

    # ⭐ SKIP nếu đã có _final.zip (đã train xong)
    final_path = agent_dir / f"sac_{building_name}_final.zip"
    if final_path.exists():
        print(f"  ⏭️  {building_name} đã có _final.zip → SKIP training")
        agents[building_name] = None
        vec_envs[building_name] = None
        continue

    # ------ Tạo vectorized environment ------
    schema_i = BUILDING_SCHEMAS[building_name]
    env_fns  = [make_env(schema_i, EPISODE_LENGTH) for _ in range(NUM_CPU)]
    vec_env  = DummyVecEnv(env_fns)
    vec_envs[building_name] = vec_env

    # ------ Kiểm tra checkpoint trung gian ------
    existing_ckpts = glob.glob(str(agent_dir / "sac_*.zip"))
    existing_ckpts = [f for f in existing_ckpts
                      if "replay_buffer" not in f and "_final" not in f and "_OLD" not in f]

    if existing_ckpts:
        latest_ckpt = max(existing_ckpts, key=lambda f: int(f.split('_')[-2]))
        print(f"  → Resume từ: {Path(latest_ckpt).name}")
        agent = SAC.load(
            latest_ckpt,
            env=vec_env,
            tensorboard_log=str(TENSORBOARD_DIR),
            device="cuda",
        )
        rb_path = latest_ckpt.replace(".zip", "_replay_buffer.pkl")
        if os.path.exists(rb_path):
            agent.load_replay_buffer(rb_path)
            print(f"  → Replay buffer loaded ({agent.replay_buffer.size():,} transitions)")
        reset_steps = False

    else:
        print(f"  → Khởi tạo model mới")
        agent = SAC(
            "MlpPolicy",
            vec_env,
            policy_kwargs=dict(net_arch=NET_ARCH),
            batch_size=BATCH_SIZE,
            buffer_size=BUFFER_SIZE,
            learning_starts=LEARNING_STARTS,
            train_freq=TRAIN_FREQ,
            gradient_steps=GRADIENT_STEPS,
            device="cuda",
            verbose=1,
            tensorboard_log=str(TENSORBOARD_DIR),
        )
        reset_steps = True

    agent._reset_steps_flag = reset_steps
    agents[building_name] = agent

# Summary
n_skip  = sum(1 for v in agents.values() if v is None)
n_train = NUM_HOUSES - n_skip
print(f"\n{'='*50}")
print(f"  Skip (đã có _final.zip) : {n_skip} agents")
print(f"  Sẽ train                : {n_train} agents")
print(f"{'='*50}")

# Print device cho agent đầu tiên còn cần train
for name in BUILDING_NAMES:
    if agents[name] is not None:
        print(f"\nDevice: {next(agents[name].policy.parameters()).device}")
        print(f"Policy network (actor):")
        print(agents[name].policy.actor)
        break
else:
    print(f"\n💡 Tất cả agents đã train xong, không cần train thêm.")


[Building_1] Đang khởi tạo...
  ⏭️  Building_1 đã có _final.zip → SKIP training

[Building_2] Đang khởi tạo...
  ⏭️  Building_2 đã có _final.zip → SKIP training

[Building_3] Đang khởi tạo...
  ⏭️  Building_3 đã có _final.zip → SKIP training

[Building_4] Đang khởi tạo...
  ⏭️  Building_4 đã có _final.zip → SKIP training

[Building_5] Đang khởi tạo...
  ⏭️  Building_5 đã có _final.zip → SKIP training

[Building_6] Đang khởi tạo...
  → Resume từ: sac_Building_6_168000_steps.zip


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



  Skip (đã có _final.zip) : 5 agents
  Sẽ train                : 1 agents

Device: cuda:0
Policy network (actor):
Actor(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (latent_pi): Sequential(
    (0): Linear(in_features=32, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (mu): Linear(in_features=256, out_features=3, bias=True)
  (log_std): Linear(in_features=256, out_features=3, bias=True)
)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
save_freq_steps = max((EPISODE_LENGTH * SAVE_EVERY_EP) // NUM_CPU, 1)
rsync_freq_steps = max((EPISODE_LENGTH * RSYNC_EVERY_EP) // NUM_CPU, 1)

print(f"Checkpoint mỗi {save_freq_steps:,} steps (~{SAVE_EVERY_EP} episodes)")
print(f"Rsync to Drive mỗi {rsync_freq_steps:,} steps (~{RSYNC_EVERY_EP} episodes)\n")

for idx, building_name in enumerate(BUILDING_NAMES):
    # ⭐ SKIP nếu agent là None (đã có _final.zip)
    if agents[building_name] is None:
        print(f"\n⏭️  [{idx+1}/{NUM_HOUSES}] Skip {building_name} (đã có _final.zip)")
        continue

    print(f"\n{'='*60}")
    print(f"  [{idx+1}/{NUM_HOUSES}] Training agent: {building_name}")
    print(f"{'='*60}")

    agent     = agents[building_name]
    agent_dir = MODEL_DIR / building_name
    drive_agent_dir = DRIVE_MODEL_DIR / building_name
    drive_agent_dir.mkdir(parents=True, exist_ok=True)

    # Callback 1: Lưu checkpoint vào local cache
    checkpoint_cb = CheckpointCallback(
        save_freq=save_freq_steps,
        save_path=str(agent_dir),
        name_prefix=f"sac_{building_name}",
        save_replay_buffer=True,
        save_vecnormalize=False,
    )

    # Callback 2: Rsync local → Drive định kỳ (background)
    rsync_cb = RsyncToDriveCallback(
        local_dir=agent_dir,
        drive_dir=drive_agent_dir,
        rsync_every_steps=rsync_freq_steps,
        verbose=1,
    )

    callbacks = CallbackList([checkpoint_cb, rsync_cb])

    agent.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=callbacks,
        tb_log_name=f"{SESSION_NAME}_{building_name}",
        reset_num_timesteps=agent._reset_steps_flag,
        log_interval=50,   # ⬆️ 1→10: giảm log spam
    )

    # ------ Lưu final model + sync ngay lên Drive ------
    final_path = agent_dir / f"sac_{building_name}_final"
    agent.save(str(final_path))
    agent.save_replay_buffer(str(agent_dir / f"sac_{building_name}_final_replay_buffer"))

    print(f"  📤 Đang sync {building_name} lên Drive...")
    subprocess.run(
        ["rsync", "-a", f"{agent_dir}/", f"{drive_agent_dir}/"],
        check=False
    )
    subprocess.run(
        ["rsync", "-a", f"{TENSORBOARD_DIR}/", f"{DRIVE_TB_DIR}/"],
        check=False
    )
    print(f"  ✓ Saved + Synced: {building_name}")

# Final sync toàn bộ
print(f"\n{'='*60}")
print("  📤 Final sync toàn bộ lên Drive...")
subprocess.run(["rsync", "-av", f"{MODEL_DIR}/", f"{DRIVE_MODEL_DIR}/"], check=False)
subprocess.run(["rsync", "-av", f"{TENSORBOARD_DIR}/", f"{DRIVE_TB_DIR}/"], check=False)
print(f"  ✅ TRAINING COMPLETE — {NUM_HOUSES} agents trained")
print(f"  📁 Drive folder: {DRIVE_MODEL_DIR}")
print(f"{'='*60}")

Checkpoint mỗi 4,200 steps (~100 episodes)
Rsync to Drive mỗi 8,400 steps (~200 episodes)


⏭️  [1/6] Skip Building_1 (đã có _final.zip)

⏭️  [2/6] Skip Building_2 (đã có _final.zip)

⏭️  [3/6] Skip Building_3 (đã có _final.zip)

⏭️  [4/6] Skip Building_4 (đã có _final.zip)

⏭️  [5/6] Skip Building_5 (đã có _final.zip)

  [6/6] Training agent: Building_6
Logging to /content/_marl_solar_cache/training_logs/marl_sac_solar_comfort_Building_6_0
  💾 Synced to Drive @ step 168,004


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  💾 Synced to Drive @ step 176,404


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 167       |
|    ep_rew_mean     | -6.67e+03 |
| time/              |           |
|    episodes        | 1050      |
|    fps             | 96        |
|    time_elapsed    | 90        |
|    total_timesteps | 176684    |
| train/             |           |
|    actor_loss      | 4.18e+03  |
|    critic_loss     | 1.37e+04  |
|    ent_coef        | 5.79      |
|    ent_coef_loss   | 0.19      |
|    learning_rate   | 0.0003    |
|    n_updates       | 46080     |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 167       |
|    ep_rew_mean     | -6.81e+03 |
| time/              |           |
|    episodes        | 1100      |
|    fps             | 97        |
|    time_elapsed    | 171       |
|    total_timesteps | 184700    |
| train/             |           |
|    actor_loss      | 4.16e+03  |
|    critic_loss    

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  📤 Đang sync Building_6 lên Drive...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  ✓ Saved + Synced: Building_6

  📤 Final sync toàn bộ lên Drive...
  ✅ TRAINING COMPLETE — 6 agents trained
  📁 Drive folder: /content/drive/MyDrive/BigData/xxxmarl_solar_comfort/models/marl_sac_solar_comfort


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Training — Huấn luyện tuần tự từng agent

Mỗi agent huấn luyện **độc lập** trên môi trường 1-nhà của mình.

**Tại sao train tuần tự (không song song)?**  
Trong notebook này, các agent train lần lượt để đơn giản hóa. Trong thực tế production (và trong FedRL), mỗi agent chạy trên **node độc lập** → hoàn toàn song song.

**Checkpoint strategy:**
$$\\text{save\\_freq\\_steps} = \\left\\lfloor \\frac{T_{ep} \\times N_{save\\_ep}}{N_{cpu}} \\right\\rfloor$$
$$= \\left\\lfloor \\frac{168 \\times 200}{2} \\right\\rfloor = 16{,}800 \\text{ steps}$$

**TensorBoard**: Mỗi agent log riêng với tên `{session}_{building_name}` để so sánh:
```bash
tensorboard --logdir training_logs/
```

## Evaluate — Đánh giá MARL trên môi trường joint (6 nhà)

Đây là bước **quan trọng nhất** của MARL: các agent **chưa bao giờ gặp nhau** trong training, nhưng giờ phải phối hợp trên môi trường thực 6-nhà.

### Cơ chế split-obs / combine-actions

```
Joint obs (246-dim)
    │
    ├─ obs[0:41]   → Agent_1 → action_1 (3-dim) ─┐
    ├─ obs[41:82]  → Agent_2 → action_2 (3-dim)  │
    ├─ obs[82:123] → Agent_3 → action_3 (3-dim)  ├─ joint_action (18-dim) → env.step()
    ├─ obs[123:164]→ Agent_4 → action_4 (3-dim)  │
    ├─ obs[164:205]→ Agent_5 → action_5 (3-dim)  │
    └─ obs[205:246]→ Agent_6 → action_6 (3-dim) ─┘
```

Công thức tổng quát:
$$o^i = o_{joint}[i \\cdot d_{obs} : (i+1) \\cdot d_{obs}], \\quad d_{obs} = 41$$
$$a_{joint} = \\text{concat}(a^1, a^2, ..., a^N), \\quad a^i = \\pi^i(o^i)$$

> Khi evaluate dùng `deterministic=True` — tắt noise Gaussian, chỉ dùng mean $\\mu_\\theta(o^i)$.

In [8]:
# ==============================================================
# 1. TẠO JOINT EVALUATION ENVIRONMENT (6 nhà)
# ==============================================================
joint_schema = copy.deepcopy(FULL_SCHEMA)
joint_schema["buildings"] = {
    name: joint_schema["buildings"][name] for name in BUILDING_NAMES
}

# Không dùng episode_time_steps → chạy toàn bộ dataset
eval_env_raw  = CityLearnEnv(joint_schema, central_agent=True)
eval_env_norm = NormalizedObservationWrapper(eval_env_raw)
eval_env      = StableBaselines3Wrapper(eval_env_norm)

# Kiểm tra kích thước
joint_obs_dim = eval_env.observation_space.shape[0]
joint_act_dim = eval_env.action_space.shape[0]
obs_per_agent = joint_obs_dim // NUM_HOUSES
act_per_agent = joint_act_dim // NUM_HOUSES

print(f"Joint observation space : {eval_env.observation_space}")
print(f"Joint action space      : {eval_env.action_space}")
print(f"Obs per agent           : {obs_per_agent}")
print(f"Act per agent           : {act_per_agent}")

# ==============================================================
# 2. TẢI FINAL AGENTS
# ==============================================================
print("\nĐang tải final agents...")
loaded_agents = {}
for building_name in BUILDING_NAMES:
    model_path = MODEL_DIR / building_name / f"sac_{building_name}_final.zip"
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model không tồn tại: {model_path}\n"
            "Hãy chạy cell Training trước."
        )
    loaded_agents[building_name] = SAC.load(str(model_path))
    print(f"  ✓ Loaded: {building_name}")

Joint observation space : Box(0.0, 1.0, (87,), float32)
Joint action space      : Box([-1. -1.  0. -1. -1.  0. -1. -1.  0. -1. -1.  0. -1. -1.  0. -1. -1.  0.], 1.0, (18,), float32)
Obs per agent           : 14
Act per agent           : 3

Đang tải final agents...
  ✓ Loaded: Building_1
  ✓ Loaded: Building_2
  ✓ Loaded: Building_3
  ✓ Loaded: Building_4
  ✓ Loaded: Building_5
  ✓ Loaded: Building_6


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
# ==============================================================
# 3. EVALUATION ROLLOUT — Mỗi agent eval trên env riêng (đúng IL)
# ==============================================================
import numpy as np

print("Đang chạy deterministic rollout cho từng agent...")
print(f"{'─'*50}")

per_agent_results = {}
all_eval_envs_raw = {}  # Giữ ref để evaluate KPIs sau

for building_name in BUILDING_NAMES:
    # Tạo env riêng cho building này (full episode, không giới hạn time_steps)
    schema_i = copy.deepcopy(FULL_SCHEMA)
    schema_i["buildings"] = {building_name: schema_i["buildings"][building_name]}

    env_raw  = CityLearnEnv(schema=schema_i, central_agent=True)
    env_norm = NormalizedObservationWrapper(env_raw)
    env_i    = StableBaselines3Wrapper(env_norm)
    all_eval_envs_raw[building_name] = env_raw

    # Verify obs dim match với agent
    expected_dim = loaded_agents[building_name].observation_space.shape[0]
    actual_dim   = env_i.observation_space.shape[0]
    if expected_dim != actual_dim:
        print(f"  ⚠ {building_name}: obs dim mismatch ({actual_dim} vs {expected_dim})")

    # Rollout
    obs, _info = env_i.reset()
    done = False
    total_reward = 0.0
    step_count = 0

    while not done:
        action, _ = loaded_agents[building_name].predict(
            obs.reshape(1, -1), deterministic=True
        )
        step_result = env_i.step(action[0])
        if len(step_result) == 5:
            obs, reward, terminated, truncated, info = step_result
            done = terminated or truncated
        else:
            obs, reward, done, info = step_result

        r = reward[0] if hasattr(reward, "__len__") else float(reward)
        total_reward += r
        step_count += 1

    per_agent_results[building_name] = {
        "total_reward": total_reward,
        "steps": step_count,
        "avg_reward": total_reward / step_count
    }
    print(f"  {building_name}: {step_count:,} steps | total_reward = {total_reward:.2f}")

print(f"{'─'*50}")
print(f"Tổng reward (6 buildings): {sum(r['total_reward'] for r in per_agent_results.values()):.2f}")
print(f"Trung bình reward/step:    {np.mean([r['avg_reward'] for r in per_agent_results.values()]):.4f}")

Đang chạy deterministic rollout cho từng agent...
──────────────────────────────────────────────────
  Building_1: 2,207 steps | total_reward = -97084.57


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Building_2: 2,207 steps | total_reward = -3730.67


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Building_3: 2,207 steps | total_reward = -1222.30


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Building_5: 2,207 steps | total_reward = -18850.57


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Building_6: 2,207 steps | total_reward = -39546.91
──────────────────────────────────────────────────
Tổng reward (6 buildings): -167369.27
Trung bình reward/step:    -12.6393


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## KPI Extraction & So sánh với Baseline

### Các KPI chính cần quan tâm

| KPI | Ý nghĩa | Tốt khi |
|---|---|---|
| `cost_total` | Chi phí điện (normalized) | < 1.0 |
| `discomfort_proportion` | % thời gian nhiệt độ ngoài setpoint | Thấp |
| `carbon_emissions_total` | Phát thải CO₂ (normalized) | < 1.0 |
| `electricity_consumption_total` | Điện tiêu thụ (normalized) | < 1.0 |
| `zero_net_energy` | Tiến tới ZNE (0 = perfect) | < 1.0 |

**Normalization**: Tất cả KPI được normalize so với **baseline không có agent** → giá trị < 1.0 nghĩa là tốt hơn baseline, > 1.0 là tệ hơn.

In [10]:
# ==============================================================
# 4. KPI EXTRACTION — Gộp từ 6 single-building envs
# ==============================================================
all_kpis = []
for building_name in BUILDING_NAMES:
    env_raw_i = all_eval_envs_raw[building_name]
    kpis_i = env_raw_i.evaluate()
    all_kpis.append(kpis_i)

# Concat tất cả
kpis = pd.concat(all_kpis, ignore_index=True)

# Pivot table
kpis_pivot = (
    kpis
    .pivot_table(index="cost_function", columns="name", values="value", aggfunc='mean')
    .round(4)
    .dropna(how="all")
)

print("=== MARL — Independent Learners KPIs ===")
display(kpis_pivot)

# Tóm tắt district-level (trung bình các nhà)
district_kpis = kpis_pivot.mean(axis=1).round(4)
print("\n=== District-level KPIs (trung bình) ===")
print(district_kpis)

=== MARL — Independent Learners KPIs ===


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


name,Building_1,Building_2,Building_3,Building_4,Building_5,Building_6,District
cost_function,,,,,,,
all_time_peak_average,NaN,NaN,NaN,NaN,NaN,NaN,0.9297
annual_normalized_unserved_energy_total,0.0100,0.0202,0.0182,0.0123,0.0167,0.0096,0.0145
carbon_emissions_total,0.6234,1.1091,0.9207,0.8233,1.0475,0.6255,0.8582
cost_total,0.5958,1.1089,0.8945,0.7923,1.0170,0.6091,0.8363
daily_one_minus_load_factor_average,NaN,NaN,NaN,NaN,NaN,NaN,1.1097
daily_peak_average,NaN,NaN,NaN,NaN,NaN,NaN,0.9563
discomfort_cold_delta_average,0.0931,0.1219,0.1106,0.2789,0.4041,0.2411,0.2083
discomfort_cold_delta_maximum,4.0356,3.0206,2.8548,4.1161,6.3374,7.1959,4.5934
discomfort_cold_delta_minimum,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000



=== District-level KPIs (trung bình) ===
cost_function
all_time_peak_average                            0.9297
annual_normalized_unserved_energy_total          0.0145
carbon_emissions_total                           0.8582
cost_total                                       0.8363
daily_one_minus_load_factor_average              1.1097
daily_peak_average                               0.9563
discomfort_cold_delta_average                    0.2083
discomfort_cold_delta_maximum                    4.5934
discomfort_cold_delta_minimum                    0.0000
discomfort_cold_proportion                       0.0303
discomfort_hot_delta_average                     1.8778
discomfort_hot_delta_maximum                     9.6499
discomfort_hot_delta_minimum                     0.0000
discomfort_hot_proportion                        0.3166
discomfort_proportion                            0.3470
electricity_consumption_total                    0.8564
monthly_one_minus_load_factor_average           

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
# ==============================================================
# 5. BẢNG SO SÁNH TOÀN DIỆN
# ==============================================================

# Kết quả từ baseline notebooks (đã có trong repo)
BASELINE_RESULTS = {
    "RBC (1 nhà)": {
        "cost_total":                       2.013,
        "discomfort_proportion":            0.984,
        "carbon_emissions_total":           2.141,
        "electricity_consumption_total":    2.121,
        "zero_net_energy":                  2.199,
        "all_time_peak_average":            1.138,
        "ramping_average":                  1.091,
    },
    "SAC Centralized (1 nhà)": {
        "cost_total":                       0.8959,
        "discomfort_proportion":            0.0657,
        "carbon_emissions_total":           0.9293,
        "electricity_consumption_total":    0.9282,
        "zero_net_energy":                  0.8210,
        "all_time_peak_average":            1.0313,
        "ramping_average":                  1.9068,
    },
    "SAC Centralized (6 nhà)": {
        "cost_total":                       0.7947,
        "discomfort_proportion":            0.5415,
        "carbon_emissions_total":           0.8016,
        "electricity_consumption_total":    0.7991,
        "zero_net_energy":                  0.7427,
        "all_time_peak_average":            0.9460,
        "ramping_average":                  1.1309,
    },
}

# Lấy KPI của MARL từ kết quả evaluate (cột District)
KEY_METRICS = [
    "cost_total",
    "discomfort_proportion",
    "carbon_emissions_total",
    "electricity_consumption_total",
    "zero_net_energy",
    "all_time_peak_average",
    "ramping_average",
]

marl_district = {}
for metric in KEY_METRICS:
    try:
        marl_district[metric] = kpis_pivot.loc[metric, "District"]
    except KeyError:
        marl_district[metric] = float("nan")

BASELINE_RESULTS["MARL IL (6 nhà) ← YOU ARE HERE"] = marl_district

# Xây dựng DataFrame so sánh
comparison_df = pd.DataFrame(BASELINE_RESULTS, index=KEY_METRICS)
comparison_df.index.name = "KPI"

# Highlight: giá trị thấp nhất trong mỗi row (tốt nhất)
def highlight_best(s):
    is_best = s == s.min()
    return ["background-color: #c8f7c5; font-weight: bold" if v else "" for v in is_best]

styled = (
    comparison_df
    .style
    .apply(highlight_best, axis=1)
    .format("{:.4f}", na_rep="N/A")
    .set_caption("So sánh các phương pháp điều khiển (giá trị thấp = tốt hơn)")
)

print("\n=== BẢNG SO SÁNH ===")
display(styled)


=== BẢNG SO SÁNH ===


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,RBC (1 nhà),SAC Centralized (1 nhà),SAC Centralized (6 nhà),MARL IL (6 nhà) ← YOU ARE HERE
KPI,,,,
cost_total,2.0130,0.8959,0.7947,0.8363
discomfort_proportion,0.9840,0.0657,0.5415,0.3470
carbon_emissions_total,2.1410,0.9293,0.8016,0.8582
electricity_consumption_total,2.1210,0.9282,0.7991,0.8564
zero_net_energy,2.1990,0.8210,0.7427,0.8006
all_time_peak_average,1.1380,1.0313,0.9460,0.9297
ramping_average,1.0910,1.9068,1.1309,1.4465


In [12]:
# ==============================================================
# 6. PHÂN TÍCH PER-BUILDING
# ==============================================================
print("=== KPI từng tòa nhà (MARL IL) ===\n")

# Chỉ lấy các cột building (không lấy District)
per_building_cols = [c for c in kpis_pivot.columns if c != "District"]
per_building_kpis = kpis_pivot[per_building_cols].loc[
    [m for m in KEY_METRICS if m in kpis_pivot.index]
]

display(per_building_kpis)

# Tính variance giữa các agent → đo độ "fair" của MARL
print("\n=== Variance giữa các tòa nhà (cao = bất bình đẳng) ===")
variance_df = per_building_kpis.var(axis=1).round(6).rename("variance")
print(variance_df.to_string())

=== KPI từng tòa nhà (MARL IL) ===



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


name,Building_1,Building_2,Building_3,Building_4,Building_5,Building_6
cost_function,,,,,,
cost_total,0.5958,1.1089,0.8945,0.7923,1.0170,0.6091
discomfort_proportion,0.8107,0.0707,0.0230,0.2003,0.3465,0.6307
carbon_emissions_total,0.6234,1.1091,0.9207,0.8233,1.0475,0.6255
electricity_consumption_total,0.6228,1.1069,0.9184,0.8233,1.0415,0.6253
zero_net_energy,0.4604,1.0232,0.9178,0.7943,1.0539,0.5539
all_time_peak_average,NaN,NaN,NaN,NaN,NaN,NaN
ramping_average,NaN,NaN,NaN,NaN,NaN,NaN



=== Variance giữa các tòa nhà (cao = bất bình đẳng) ===
cost_function
cost_total                       0.044349
discomfort_proportion            0.099668
carbon_emissions_total           0.042638
electricity_consumption_total    0.041986
zero_net_energy                  0.060817
all_time_peak_average                 NaN
ramping_average                       NaN


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Kết luận & Bước tiếp theo

### MARL vs Centralized SAC — Phân tích

| Góc độ | MARL Independent Learners | SAC Centralized |
|---|---|---|
| **Observation space** | 41-dim/agent (nhỏ, hội tụ nhanh) | 246-dim (lớn, khó train) |
| **Privacy** | Mỗi nhà giữ data riêng | Phải share toàn bộ obs |
| **Scalability** | Thêm nhà = thêm agent mới | Phải train lại từ đầu |
| **Coordination** | Không có (Independent) | Có (joint policy) |
| **Phù hợp với FedRL** | ✓ Tự nhiên | ✗ Không |

### Non-stationarity Problem

MARL IL có điểm yếu cốt lõi: khi agent $i$ đang học, môi trường của nó **không dừng (non-stationary)** vì agent $j$ cũng đang thay đổi:

$$P(s_{t+1} | s_t, a^i_t) \\neq P(s_{t+1} | s_t, a^i_t, a^{-i}_t)$$

Agent $i$ không quan sát được $a^{-i}_t$ → vi phạm Markov assumption của SAC.

### Bước tiếp theo → Federated RL

```
MARL IL (hiện tại)
    ↓  + periodic weight aggregation
FedRL
    ↓
w_global = Σ (n_i/n) × w_i   [FedAvg]
```

Giải pháp non-stationarity trong FedRL:
- **Synchronous aggregation**: Tất cả agent đồng bộ hóa sau mỗi $K$ episodes
- **Shared global policy**: $w_{global}$ mang thông tin của tất cả buildings
- **Local fine-tuning**: Mỗi agent fine-tune từ $w_{global}$ → personalization

In [13]:
from google.colab import runtime

# Sau khi chạy xong, lệnh này sẽ hủy session ngay lập tức
runtime.unassign()